# Laboratorio de Detección Facial con Amazon Rekognition


## Objetivos del laboratorio:
1. Crear una colección de rostros en Amazon Rekognition
2. Indexar una imagen de referencia en la colección
3. Buscar coincidencias de rostros en nuevas imágenes
4. Visualizar los resultados con cuadros delimitadores

---

## Paso 1: Importación de Librerías

Importamos todas las librerías necesarias:
- **matplotlib**: Para visualización de imágenes
- **skimage**: Para manipulación y transformación de imágenes
- **boto3**: SDK de AWS para Python
- **numpy**: Para operaciones numéricas
- **PIL**: Para dibujar sobre imágenes

In [ ]:
# Importación de librerías necesarias
from skimage import io
from skimage.transform import rescale
from matplotlib import pyplot as plt

import boto3

import numpy as np

from PIL import Image, ImageDraw, ImageColor, ImageOps

# Para manejo de errores de AWS
from botocore.exceptions import ClientError

print("✅ Librerías importadas correctamente")

## Paso 2: Crear una Colección en Amazon Rekognition

Una **colección** es un contenedor que almacena información sobre los rostros detectados.
Cada rostro se representa como un vector de características (face embedding).

**Nota importante:** Solo ejecutar este paso una vez. Si la colección ya existe, se generará un error.

In [ ]:
# Crear cliente de Amazon Rekognition
client = boto3.client('rekognition')

# Definir el ID de la colección
collection_id = 'Collection'

# Crear la colección
try:
    response = client.create_collection(CollectionId=collection_id)
    print('✅ Colección creada exitosamente')
    print('Collection ARN: ' + response['CollectionArn'])
    print('Status Code: ' + str(response['StatusCode']))
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceAlreadyExistsException':
        print('⚠️ La colección ya existe, continuando...')
    else:
        raise e

print('Done...')

## Paso 3: Cargar la Imagen de Referencia

Cargamos la imagen `mum.jpg` que será nuestro rostro de referencia para búsquedas posteriores.

In [ ]:
# Nombre del archivo de imagen de referencia
filename = "mum.jpg"

# Cargar la imagen usando skimage
faceimage = io.imread(filename)

# Mostrar la imagen
plt.figure(figsize=(8, 6))
plt.imshow(faceimage)
plt.title('Imagen de Referencia: ' + filename)
plt.axis('off')
plt.show()

# Mostrar dimensiones de la imagen
print(f"📐 Dimensiones de la imagen: {faceimage.shape}")

### Redimensionar imagen (si es necesario)

Amazon Rekognition tiene un límite de **4096 x 4096 píxeles**. Si la imagen es más grande, debemos redimensionarla.

In [ ]:
# Verificar si necesita redimensionarse
height, width = faceimage.shape[:2]

if height > 4096 or width > 4096:
    print("⚠️ Imagen demasiado grande, redimensionando...")
    # Escalar al 50%
    faceimage = rescale(faceimage, 0.50, mode='constant', channel_axis=2)
    # Guardar la imagen redimensionada
    io.imsave(filename, (faceimage * 255).astype(np.uint8))
    print(f"✅ Imagen redimensionada a: {faceimage.shape}")
else:
    print(f"✅ Imagen dentro de los límites: {width}x{height} píxeles")

## Paso 4: Indexar el Rostro en la Colección

Usamos `index_faces` para agregar el rostro a nuestra colección. Esta operación:
- Detecta rostros en la imagen
- Extrae características faciales (embeddings)
- Almacena la información en la colección

In [ ]:
# ID externo para identificar la imagen
externalimageid = filename

# Abrir la imagen y enviarla a Rekognition
with open(filename, 'rb') as fimage:
    response = client.index_faces(
        CollectionId=collection_id,
        Image={'Bytes': fimage.read()},
        ExternalImageId=externalimageid,
        MaxFaces=1,                    # Solo indexar 1 rostro
        QualityFilter="AUTO",          # Filtro automático de calidad
        DetectionAttributes=['ALL']    # Detectar todos los atributos
    )

# Mostrar resultados
print('📊 Resultados para ' + filename)
print('\n✅ Rostros indexados:')
for faceRecord in response['FaceRecords']:
    print('  • Face ID: ' + faceRecord['Face']['FaceId'])
    print('  • Ubicación: {}'.format(faceRecord['Face']['BoundingBox']))
    print('  • Confianza: {:.2f}%'.format(faceRecord['Face']['Confidence']))

if response['UnindexedFaces']:
    print('\n⚠️ Rostros NO indexados:')
    for unindexedFace in response['UnindexedFaces']:
        print('  • Ubicación: {}'.format(unindexedFace['FaceDetail']['BoundingBox']))
        print('  • Razones:')
        for reason in unindexedFace['Reasons']:
            print('    - ' + reason)
else:
    print('\n✅ Todos los rostros fueron indexados correctamente')

## Paso 5: Visualizar el Cuadro Delimitador

Dibujamos un rectángulo alrededor del rostro detectado usando las coordenadas del **BoundingBox**.

El BoundingBox contiene valores normalizados (0-1) que representan:
- `Left`: Posición X del borde izquierdo
- `Top`: Posición Y del borde superior
- `Width`: Ancho del rectángulo
- `Height`: Alto del rectángulo

In [ ]:
# Abrir la imagen con PIL
img = Image.open(filename)
imgWidth, imgHeight = img.size

# Crear objeto para dibujar
draw = ImageDraw.Draw(img)

# Dibujar cuadro delimitador para cada rostro
for faceRecord in response['FaceRecords']:
    box = faceRecord['Face']['BoundingBox']
    
    # Convertir coordenadas normalizadas a píxeles
    left = imgWidth * box['Left']
    top = imgHeight * box['Top']
    width = imgWidth * box['Width']
    height = imgHeight * box['Height']
    
    # Definir los puntos del rectángulo
    points = (
        (left, top),
        (left + width, top),
        (left + width, top + height),
        (left, top + height),
        (left, top)  # Cerrar el rectángulo
    )
    
    # Dibujar el rectángulo en verde
    draw.line(points, fill='#00d400', width=15)

# Mostrar la imagen con el cuadro delimitador
plt.figure(figsize=(10, 8))
plt.imshow(img)
plt.title('Rostro Detectado con Cuadro Delimitador')
plt.axis('off')
plt.show()

## Paso 6: Listar Rostros en la Colección

Verificamos qué rostros están almacenados en nuestra colección usando `list_faces`.

In [ ]:
# Configuración de paginación
maxResults = 2
faces_count = 0
tokens = True

# Primera llamada a list_faces
response_list = client.list_faces(
    CollectionId=collection_id,
    MaxResults=maxResults
)

print('📋 Rostros en la colección: ' + collection_id)
print('=' * 50)

# Iterar sobre todos los resultados (con paginación)
while tokens:
    faces = response_list['Faces']
    
    for face in faces:
        faces_count += 1
        print(f"\n🔹 Rostro #{faces_count}")
        print(f"   Face ID: {face['FaceId']}")
        print(f"   External ID: {face.get('ExternalImageId', 'N/A')}")
        print(f"   Confianza: {face.get('Confidence', 'N/A'):.2f}%")
    
    # Verificar si hay más páginas
    if 'NextToken' in response_list:
        nextToken = response_list['NextToken']
        response_list = client.list_faces(
            CollectionId=collection_id,
            NextToken=nextToken,
            MaxResults=maxResults
        )
    else:
        tokens = False

print(f"\n📊 Total de rostros en la colección: {faces_count}")

## Paso 7: Buscar un Rostro en una Nueva Imagen

Ahora buscaremos si el rostro indexado aparece en una imagen diferente (`target.jpg`) usando `search_faces_by_image`.

In [ ]:
# Cargar imagen objetivo
targetfilename = "target.jpg"

targetimage = Image.open(targetfilename)

# Mostrar la imagen objetivo
plt.figure(figsize=(10, 8))
plt.imshow(targetimage)
plt.title('Imagen Objetivo: ' + targetfilename)
plt.axis('off')
plt.show()

print(f"📐 Dimensiones: {targetimage.size}")

In [ ]:
# Parámetros de búsqueda
threshold = 70      # Umbral de similitud mínimo (70%)
maxFaces = 2        # Máximo de coincidencias a retornar

# Realizar búsqueda
with open(targetfilename, 'rb') as timage:
    response2 = client.search_faces_by_image(
        CollectionId=collection_id,
        Image={'Bytes': timage.read()},
        FaceMatchThreshold=threshold,
        MaxFaces=maxFaces
    )

# Mostrar resultados
faceMatches = response2['FaceMatches']

if faceMatches:
    print('🎯 ¡Coincidencias encontradas!')
    print('=' * 50)
    for i, match in enumerate(faceMatches, 1):
        print(f"\n✅ Coincidencia #{i}")
        print(f"   Face ID: {match['Face']['FaceId']}")
        print(f"   Similitud: {match['Similarity']:.2f}%")
        print(f"   Imagen original: {match['Face']['ExternalImageId']}")
else:
    print('❌ No se encontraron coincidencias con el umbral establecido')

## Paso 8: Dibujar Cuadro Delimitador en la Imagen Objetivo

Visualizamos dónde se encontró el rostro coincidente en la imagen objetivo.

In [ ]:
# Recargar la imagen para dibujar
targetimage = Image.open(targetfilename)
imgWidth, imgHeight = targetimage.size

# Crear objeto para dibujar
draw = ImageDraw.Draw(targetimage)

# Obtener el BoundingBox del rostro buscado
box = response2['SearchedFaceBoundingBox']

# Convertir coordenadas normalizadas a píxeles
left = imgWidth * box['Left']
top = imgHeight * box['Top']
width = imgWidth * box['Width']
height = imgHeight * box['Height']

# Definir los puntos del rectángulo
points = (
    (left, top),
    (left + width, top),
    (left + width, top + height),
    (left, top + height),
    (left, top)
)

# Dibujar el rectángulo en verde
draw.line(points, fill='#00d400', width=15)

# Mostrar resultado
plt.figure(figsize=(10, 8))
plt.imshow(targetimage)
plt.title('Rostro Encontrado en Imagen Objetivo')
plt.axis('off')
plt.show()

print("✅ Cuadro delimitador dibujado exitosamente")

## Paso 9: Limpiar Recursos - Eliminar la Colección

Es importante eliminar la colección cuando ya no la necesitemos para evitar cargos innecesarios.

In [ ]:
print('🗑️ Intentando eliminar colección: ' + collection_id)

try:
    response_delete = client.delete_collection(CollectionId=collection_id)
    status_code = response_delete['StatusCode']
    print('\n✅ ¡Colección eliminada exitosamente!')
    print(f'Status Code: {status_code}')
    
except ClientError as e:
    if e.response['Error']['Code'] == 'ResourceNotFoundException':
        print(f'\n⚠️ La colección "{collection_id}" no fue encontrada')
    else:
        print(f'\n❌ Error: {e.response["Error"]["Message"]}')
    status_code = e.response['ResponseMetadata']['HTTPStatusCode']

print('\n🎉 ¡Laboratorio completado!')

---

# ¡Felicitaciones! 🎉

Has completado exitosamente el laboratorio de detección facial con Amazon Rekognition.

## Resumen de lo aprendido:
- ✅ Crear y administrar colecciones de rostros
- ✅ Indexar rostros usando `index_faces`
- ✅ Buscar rostros con `search_faces_by_image`
- ✅ Visualizar resultados con cuadros delimitadores
- ✅ Limpiar recursos de AWS